# Demo 02 — Evaluation and Statistical Validation

**Class 01 · D4a-D4g** (slides: `slides/01_foundations.md`; lab: Part F)

**Example 2** is the breast cancer data (`datasets/breast_cancer.csv.zst`, 569 patients, 30 features), read with polars.
The class to find is **malignant**: the *positive* class, 212 patients. The file codes benign as 1, so **we recode: `y = 1` means malignant** (here, in the lab and in the guide), `y = 0` benign (357 patients).
A decision tree and logistic regression are black boxes here; they only give predictions. The question is: **how do we know whether a model is good?**
The decision tree of this demo is **unpruned** (it memorizes its training set: train accuracy 1.000); the lab's tree has `max_depth=3`, so the numbers of the two differ on purpose.

**Live in the class (about 6 minutes, DEMO D4):**

* **D4a** Hold out data: training against new data, and the size of the test set.
* **D4b** A single test score is a random variable (50 splits).
* **D4c** Cross-validation, the standard error and the confidence interval; the choice of $k$; a second dataset.
* **D4d** A baseline, the accuracy paradox, the confusion matrix and its metrics; the imbalance sweep.

**Read in the lab (Part F, no live demo):** the deck frames of the two tests quote their numbers.

* **D4e** Comparing two models: naive against corrected t-test; how often does each cry wolf?
* **D4f** McNemar's test on one test set, with the exact binomial.
* **D4g** Hypothesis testing from scratch: 10 tosses, 9 heads; a classifier against the baseline; a permutation test.

Every cell that prints a number of a slide starts with `slide: <frame title>`.

In [ ]:
import math

import numpy as np
import polars as pl
from joblib import Parallel, delayed
from scipy import stats
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits

from faa import plotlib

plotlib.setup(module="01")
BLUE, ORANGE, GREEN, RED, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 42
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_width_chars(160)

# dataset -> (target column, value of the column that is the positive class: it becomes y = 1)
DATASETS = {"breast_cancer": ("target", 0), "heart_disease": ("num", ">50_1"), "german_credit": ("class", "bad")}


def load_xy(name: str):
    """Features as floats (text columns one-hot encoded) and the target with `y = 1` for the positive class (malignant, disease, bad credit)."""
    target, positive = DATASETS[name]
    df = pl.read_csv(f"../../datasets/{name}.csv.zst")
    y = (df[target] == positive).cast(pl.Int8).to_numpy()
    feats = df.drop(target)
    text = [c for c, t in feats.schema.items() if t == pl.String]
    if text:
        feats = feats.to_dummies(columns=text, drop_first=True)
    return feats.cast(pl.Float64).to_numpy(), y


def tree_model(seed=SEED, **kw):
    return make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(random_state=seed, **kw))


def logreg_model():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000))


def show(table: pl.DataFrame, digits: int = 4) -> None:
    """Print a table with rounded floats."""
    print(table.with_columns(pl.col(pl.Float64).round(digits)))


def slide(title: str) -> None:
    print(f"--- slide: {title} ---")


def mean_ci(scores, confidence=0.95):
    """Mean, standard deviation (n - 1), standard error, t quantile and half-width of the t interval of the scores."""
    s = np.asarray(scores, dtype=float)
    k = len(s)
    sd = s.std(ddof=1)
    se = sd / np.sqrt(k)
    t = stats.t.ppf((1 + confidence) / 2, k - 1)
    return {"mean": s.mean(), "sd": sd, "se": se, "t": t, "half": t * se}


X, y = load_xy("breast_cancer")
print(
    f"breast cancer: {X.shape[0]} patients, {X.shape[1]} features; malignant (y = 1) = {np.sum(y == 1)}, benign (y = 0) = {np.sum(y == 0)}"
)

## D4a — Hold out data

Score a model on the data it was trained on, then on data it has never seen.
The split below (25% test, stratified, seed 42) is **the** split of the rest of the demo: the confusion matrix, McNemar's counts and the test of the coin all use it.

### Training against new data (slide: Training Loss Is Not the Goal; plot `d4_train_vs_new`)

In [ ]:
# slide: Training Loss Is Not the Goal; Hold Out Data: Training, Validation, and Test Sets
slide("Training Loss Is Not the Goal; Hold Out Data")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
print(
    f"n_train = {len(y_train)}, n_test = {len(y_test)}; malignant in train = {np.sum(y_train == 1)}, in test = {np.sum(y_test == 1)}"
)
tree = tree_model().fit(X_train, y_train)
logreg = logreg_model().fit(X_train, y_train)
acc = pl.DataFrame(
    {
        "model": ["decision tree", "logistic regression"],
        "train_accuracy": [tree.score(X_train, y_train), logreg.score(X_train, y_train)],
        "test_accuracy": [tree.score(X_test, y_test), logreg.score(X_test, y_test)],
    }
)
show(acc, 3)
print(f"tree: {int(round(tree.score(X_test, y_test) * len(y_test)))} of {len(y_test)} test patients correct")

In [ ]:
# plot d4_train_vs_new
fg, ax = plotlib.fig("full", 1.6)
xs, w = np.arange(2), 0.3
for j, (col, colour, lab) in enumerate(
    [("train_accuracy", BLUE, "training data"), ("test_accuracy", ORANGE, "new data (test set)")]
):
    bars = ax.bar(xs + (j - 0.5) * w, acc[col], w, color=colour, label=lab)
    for rect, v in zip(bars, acc[col]):
        ax.annotate(
            f"{v:.3f}", (rect.get_x() + w / 2, v), (0, 2), textcoords="offset points", ha="center", fontsize=7.5
        )
ax.set(xticks=xs, xticklabels=acc["model"], ylim=(0.8, 1.04), ylabel="accuracy")
ax.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=2, borderaxespad=0.1, columnspacing=1.0, handletextpad=0.3
)
ax.grid(axis="x", visible=False)
plotlib.save(fg, "d4_train_vs_new")

### Sweep: how much data to hold out? (no slide)

Mean over 20 random splits for each test fraction. A small test set gives a noisy score; a large one leaves too little to train on.

In [ ]:
rows = []
for frac in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    tr, te = [], []
    for s in range(20):
        Xa, Xb, ya, yb = train_test_split(X, y, test_size=frac, stratify=y, random_state=s)
        m = tree_model().fit(Xa, ya)
        tr.append(m.score(Xa, ya))
        te.append(m.score(Xb, yb))
    rows.append(
        {
            "test_fraction": frac,
            "n_test": len(yb),
            "train_acc": np.mean(tr),
            "test_acc": np.mean(te),
            "test_acc_sd": np.std(te, ddof=1),
        }
    )
sweep_frac = pl.DataFrame(rows)
show(sweep_frac, 3)

In [ ]:
# plot d4_testfrac (extra)
fg, ax = plotlib.fig("full", 1.6)
pct = sweep_frac["test_fraction"] * 100
ax.plot(pct, sweep_frac["train_acc"], color=BLUE, marker="o", ms=3, label="training accuracy")
ax.plot(pct, sweep_frac["test_acc"], color=ORANGE, marker="s", ms=3, label="test accuracy")
ax.fill_between(
    pct,
    sweep_frac["test_acc"] - sweep_frac["test_acc_sd"],
    sweep_frac["test_acc"] + sweep_frac["test_acc_sd"],
    color=ORANGE,
    alpha=0.2,
    lw=0,
)
ax.set(xlabel="test set (% of the data)", ylabel="accuracy", ylim=(0.8, 1.02))
ax.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=2, borderaxespad=0.1, columnspacing=1.0, handletextpad=0.3
)
plotlib.save(fg, "d4_testfrac")

## D4b — A single test score is a random variable

Repeat the same experiment with 50 different random train/test splits (25% test).

### Fifty splits (slide: One Split Is Not Enough; plot `d4_splits`)

In [ ]:
# slide: One Split Is Not Enough
slide("One Split Is Not Enough")
acc_tree, acc_lr = [], []
for s in range(50):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, stratify=y, random_state=s)
    acc_tree.append(tree_model().fit(Xa, ya).score(Xb, yb))
    acc_lr.append(logreg_model().fit(Xa, ya).score(Xb, yb))
scores_tree, scores_lr = np.array(acc_tree), np.array(acc_lr)
for name, sc in [("decision tree", scores_tree), ("logistic regression", scores_lr)]:
    print(
        f"{name:20s} test accuracy over 50 splits: min = {sc.min():.3f}, max = {sc.max():.3f}, mean = {sc.mean():.3f}, std = {sc.std(ddof=1):.3f}"
    )
print(f"the tree's range max - min = {100 * (scores_tree.max() - scores_tree.min()):.1f} percentage points")

In [ ]:
# plot d4_splits
fg, ax = plotlib.fig("full", 1.5)
ax.hist(scores_tree, bins=np.arange(0.86, 1.0, 0.01), color=BLUE, edgecolor="white", lw=0.5)
ax.axvline(scores_tree.mean(), color=RED, ls="--", lw=1.2)
ax.annotate(
    f"mean {scores_tree.mean():.3f}",
    (scores_tree.mean(), ax.get_ylim()[1]),
    (-4, -8),
    textcoords="offset points",
    ha="right",
    color=RED,
    fontsize=7.5,
)
ax.set(xlabel="test accuracy of one split", ylabel="number of splits")
plotlib.save(fg, "d4_splits")

## D4c — k-fold cross-validation and confidence intervals

Every example is used for testing exactly once. We report a **mean with its uncertainty**, not one number.

### Ten folds by hand (slides: Variance and Standard Deviation, Summarizing the Folds, Worked Example: Confidence Interval; plots `d4_variance`, `d4_folds`, `d4_ci`)

In [ ]:
# slide: Variance and Standard Deviation; Summarizing the Folds; Worked Example: Confidence Interval
slide("Worked Example: Confidence Interval (decision tree, 10-fold CV)")
K = 10
cv = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
folds = list(cv.split(X, y))
s_tree = cross_val_score(tree_model(), X, y, cv=cv)
s_lr = cross_val_score(logreg_model(), X, y, cv=cv)
mean = s_tree.mean()
dev = s_tree - mean
ci = mean_ci(s_tree)
show(
    pl.DataFrame(
        {
            "fold": range(1, K + 1),
            "n_test": [len(te) for _, te in folds],
            "s_i": s_tree,
            "s_i - mean": dev,
            "(s_i - mean)^2": dev**2,
        }
    ),
    5,
)
print(f"k = {K}, sum of s_i = {s_tree.sum():.4f}, mean = {mean:.4f}")
print(f"sum of squared deviations = {np.sum(dev**2):.6f}")
print(f"variance with n - 1 = {np.sum(dev**2) / (K - 1):.6f}   (divisor n would give {np.sum(dev**2) / K:.6f})")
print(f"s = {ci['sd']:.4f}; SE = s / sqrt(k) = {ci['sd']:.4f} / {np.sqrt(K):.4f} = {ci['se']:.4f}")
print(f"t quantile (97.5%, {K - 1} degrees of freedom) = {ci['t']:.3f}; half-width = t x SE = {ci['half']:.4f}")
print(f"95% CI = {mean:.4f} +- {ci['half']:.4f} = [{mean - ci['half']:.4f}, {mean + ci['half']:.4f}]")
print(
    f"pooled accuracy (all {len(y)} predictions) = {np.sum(s_tree * [len(te) for _, te in folds]) / len(y):.4f}; mean of the fold scores = {mean:.4f}"
)
mean_lr = s_lr.mean()
ci_lr = mean_ci(s_lr)
print(
    f"logistic regression: mean = {mean_lr:.4f}, s = {ci_lr['sd']:.4f}, SE = {ci_lr['se']:.4f}, 95% CI = {mean_lr:.4f} +- {ci_lr['half']:.4f}"
)

**Why $n - 1$?** The mean is computed from the same values, so the deviations around it are smaller than around the true mean.
Dividing by $n$ underestimates the variance, dividing by $n - 1$ corrects it on average (simulation below).

In [ ]:
# slide: Variance and Standard Deviation (n - 1 bullet)
slide("Variance and Standard Deviation: n - 1")
rng = np.random.default_rng(SEED)
draws = rng.normal(0, 1, size=(100_000, 5))
print("100 000 samples of 5 values from N(0, 1), true variance = 1")
print(f"mean of the variance with divisor n     = {draws.var(axis=1, ddof=0).mean():.3f}   (expected (n-1)/n = 0.8)")
print(f"mean of the variance with divisor n - 1 = {draws.var(axis=1, ddof=1).mean():.3f}   (unbiased)")

In [ ]:
# plot d4_variance: the fold scores of two models, with the deviation from the mean and the band of one s
fg, axs = plotlib.fig("full", 1.6, ncols=2, sharey=True)
for ax, sc, name, colour in [(axs[0], s_tree, "decision tree", BLUE), (axs[1], s_lr, "logistic regression", ORANGE)]:
    m, sd = sc.mean(), sc.std(ddof=1)
    ax.axhspan(m - sd, m + sd, color=colour, alpha=0.15, lw=0)
    ax.axhline(m, color=colour, lw=1.2)
    xs = np.arange(1, K + 1)
    ax.vlines(xs, m, sc, color=RED, lw=1)
    ax.scatter(xs, sc, s=14, color="k", zorder=4)
    ax.set_title(f"{name}\n$\\bar{{s}}$ = {m:.3f}, $s$ = {sd:.3f}", fontsize=7.5)
    ax.set(xlabel="fold", xticks=[1, 5, 10])
axs[0].set_ylabel("fold accuracy $s_i$")
plotlib.save(fg, "d4_variance")

In [ ]:
# plot d4_folds: the ten fold scores of the tree, the mean and its 95% interval
fg, ax = plotlib.fig("full", 1.6)
xs = np.arange(1, K + 1)
ax.axhspan(mean - ci["half"], mean + ci["half"], color=BLUE, alpha=0.18, lw=0)
ax.axhline(mean, color=BLUE, lw=1.2)
ax.scatter(xs, s_tree, s=16, color="k", zorder=4)
for xi, v in zip(xs, s_tree):
    ax.annotate(
        f"{v:.3f}",
        (xi, v),
        (0, 5 if v >= mean else -5),
        textcoords="offset points",
        ha="center",
        va="bottom" if v >= mean else "top",
        fontsize=7,
    )
ax.set_title(f"mean {mean:.3f} (line), 95% CI $\\pm${ci['half']:.3f} (band)", fontsize=7.5, color=BLUE, loc="left")
ax.set(xlabel="fold", ylabel="fold accuracy $s_i$", xticks=xs, xlim=(0.4, K + 0.6), ylim=(0.86, 1.0))
plotlib.save(fg, "d4_folds")

### Sweep: the choice of $k$ and a second dataset (plots `d4_ci`, `d4_ci_datasets`)

The same code on `breast_cancer` and `heart_disease` (303 patients, 13 features, `>50_1` = disease). A t interval on fold scores is an approximation: the folds share training data.

In [ ]:
rows = []
for name in ("breast_cancer", "heart_disease"):
    Xd, yd = load_xy(name)
    for k in (2, 5, 10, 20):
        sc = cross_val_score(tree_model(), Xd, yd, cv=StratifiedKFold(k, shuffle=True, random_state=SEED))
        rows.append({"dataset": name, "k": k, **mean_ci(sc)})
sweep_k = pl.DataFrame(rows)
show(sweep_k, 4)

In [ ]:
# plot d4_ci: the interval of the tree on breast cancer for k = 2, 5, 10, 20
bc = sweep_k.filter(pl.col("dataset") == "breast_cancer")
fg, ax = plotlib.fig("col", 1.7)
ys = np.arange(len(bc))
ax.errorbar(bc["mean"], ys, xerr=bc["half"], fmt="o", color=BLUE, ms=3.5, capsize=2.5, lw=1.2)
ax.set(
    yticks=ys,
    yticklabels=[f"$k$ = {k}\n$\\pm${h:.3f}" for k, h in zip(bc["k"], bc["half"])],
    xlabel="accuracy: mean and 95% CI",
    xlim=(0.7, 1.12),
)
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
plotlib.save(fg, "d4_ci")

In [ ]:
# plot d4_ci_datasets (extra): the same code on two datasets
fg, axs = plotlib.fig("full", 1.6, ncols=2)
for ax, name, colour in zip(axs, ("breast_cancer", "heart_disease"), (BLUE, ORANGE)):
    sub = sweep_k.filter(pl.col("dataset") == name)
    ys = np.arange(len(sub))
    ax.errorbar(sub["mean"], ys, xerr=sub["half"], fmt="o", color=colour, ms=3.5, capsize=2.5, lw=1.2)
    ax.set(
        yticks=ys,
        yticklabels=[f"$k$ = {k}" for k in sub["k"]],
        xlabel="accuracy: mean and 95% CI",
        title=name.replace("_", " "),
    )
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
plotlib.save(fg, "d4_ci_datasets")

**Reading the sweep.** The interval does not shrink steadily with $k$: more folds mean more scores but each fold is smaller and the scores are correlated, so the formula
$s/\sqrt{k}$ becomes optimistic. A second dataset (303 patients) has a much wider interval: the width is set by $n$ and by the noise of the problem.

## D4d — Baselines, the accuracy paradox and the confusion matrix

A "dumb" model that always predicts the majority class is the **minimum bar** any model must beat.

### Baseline (slide: Always Compare with a Baseline; plot `d4_baseline`)

In [ ]:
# slide: Always Compare with a Baseline
slide("Always Compare with a Baseline")
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
cv_scores = {
    "majority-class baseline": cross_val_score(DummyClassifier(strategy="most_frequent"), X, y, cv=cv10),
    "decision tree": cross_val_score(tree_model(), X, y, cv=cv10),
    "logistic regression": cross_val_score(logreg_model(), X, y, cv=cv10),
}
for name, sc in cv_scores.items():
    c = mean_ci(sc)
    print(f"{name:24s} 10-fold accuracy = {c['mean']:.4f} +- {c['half']:.4f} (95% CI), s = {c['sd']:.4f}")
print(f"share of the majority class (benign) = {(y == 0).mean():.4f}")

In [ ]:
# plot d4_baseline
fg, ax = plotlib.fig("full", 1.6)
rng = np.random.default_rng(SEED)
for j, (name, sc), colour in zip(range(3), cv_scores.items(), (GREY, BLUE, ORANGE)):
    ax.scatter(j + rng.uniform(-0.12, 0.12, len(sc)), sc, s=10, color=colour, alpha=0.85, zorder=3)
    m = sc.mean()
    ax.hlines(m, j - 0.3, j + 0.3, color="k", lw=1.4, zorder=4)
    ax.annotate(f"{m:.3f}", (j + 0.32, m), (2, 0), textcoords="offset points", va="center", fontsize=7.5)
ax.set(
    xticks=range(3), xticklabels=[n.replace(" ", "\n", 1) for n in cv_scores], ylabel="fold accuracy", xlim=(-0.5, 2.7)
)
ax.grid(axis="x", visible=False)
plotlib.save(fg, "d4_baseline")

### The confusion matrix of the tree on the test set of D4a (slides: The Confusion Matrix; Precision, Recall, and F1)

The positive class is **malignant**. $TP$: malignant and called malignant; $FP$: benign called malignant; $FN$: malignant called benign; $TN$: benign called benign.

In [ ]:
# slide: The Confusion Matrix; Precision, Recall, and F1
slide("The Confusion Matrix; Precision, Recall, and F1")


def confusion(y_true, y_pred):
    """TP, FP, FN, TN with y = 1 (malignant) as the positive class."""
    t, p = y_true == 1, y_pred == 1
    return int(np.sum(t & p)), int(np.sum(~t & p)), int(np.sum(t & ~p)), int(np.sum(~t & ~p))


def metrics(tp, fp, fn, tn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"accuracy": (tp + tn) / (tp + fp + fn + tn), "precision": precision, "recall": recall, "f1": f1,
            "balanced_accuracy": (recall + tn / (tn + fp)) / 2}  # fmt: skip


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
tree_pred = tree_model().fit(X_train, y_train).predict(X_test)
tp, fp, fn, tn = confusion(y_test, tree_pred)
print(f"decision tree on the {len(y_test)} test patients (positive = malignant):")
print("                    predicted malignant   predicted benign")
print(f"actual malignant    TP = {tp:3d}              FN = {fn:3d}")
print(f"actual benign       FP = {fp:3d}              TN = {tn:3d}")
mt = metrics(tp, fp, fn, tn)
print(f"accuracy  = (TP + TN) / n = ({tp} + {tn}) / {tp + fp + fn + tn} = {mt['accuracy']:.4f}")
print(f"precision = TP / (TP + FP) = {tp} / {tp + fp} = {mt['precision']:.4f}")
print(f"recall    = TP / (TP + FN) = {tp} / {tp + fn} = {mt['recall']:.4f}")
print(
    f"F1 = 2 P R / (P + R) = 2 x {mt['precision']:.4f} x {mt['recall']:.4f} / ({mt['precision']:.4f} + {mt['recall']:.4f}) = {mt['f1']:.4f}   (= 2 TP / (2 TP + FP + FN) = {2 * tp / (2 * tp + fp + fn):.4f})"
)
check = (
    precision_score(y_test, tree_pred),
    recall_score(y_test, tree_pred),
    f1_score(y_test, tree_pred),
)
print(
    f"scikit-learn (pos_label = 1, the default) agrees: precision = {check[0]:.4f}, recall = {check[1]:.4f}, F1 = {check[2]:.4f}"
)

### The accuracy paradox: thin the positive class to 5% (slide: The Accuracy Paradox; plot `d4_paradox`)

Keep all 357 benign patients and a random sample of malignant ones, so that malignant is about 5% of the data.

In [ ]:
# slide: Always Compare with a Baseline; The Accuracy Paradox
slide("The Accuracy Paradox (malignant thinned to about 5%)")


def thin(X_all, y_all, share, rng):
    """All benign patients and enough malignant ones for the given share of malignant."""
    benign, malignant = np.flatnonzero(y_all == 0), np.flatnonzero(y_all == 1)
    n_mal = round(share / (1 - share) * len(benign))
    keep = np.concatenate([benign, rng.choice(malignant, n_mal, replace=False)])
    return X_all[keep], y_all[keep]


rng = np.random.default_rng(SEED)
X5, y5 = thin(X, y, 0.05, rng)
print(
    f"thinned data: {len(y5)} patients, malignant = {np.sum(y5 == 1)} ({100 * (y5 == 1).mean():.2f}%), benign = {np.sum(y5 == 0)}"
)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
models5 = {
    "baseline": DummyClassifier(strategy="most_frequent"),
    "decision tree": tree_model(),
    "logistic regression": logreg_model(),
}
rows = []
for name, m in models5.items():
    pred = cross_val_predict(m, X5, y5, cv=cv5)  # one out-of-fold prediction per patient
    tp, fp, fn, tn = confusion(y5, pred)
    rows.append({"model": name, "TP": tp, "FP": fp, "FN": fn, "TN": tn, **metrics(tp, fp, fn, tn)})
paradox = pl.DataFrame(rows)
print(f"out-of-fold predictions of 5-fold CV for all {len(y5)} patients, {np.sum(y5 == 1)} malignant")
show(paradox, 3)

In [ ]:
# plot d4_paradox
fg, ax = plotlib.fig("full", 1.8)
names = ["accuracy", "balanced_accuracy", "recall", "f1"]
xs, w = np.arange(len(names)), 0.26
for j, (row, colour) in enumerate(zip(paradox.iter_rows(named=True), (GREY, BLUE, ORANGE))):
    vals = [row[k] for k in names]
    bars = ax.bar(xs + (j - 1) * w, vals, w, color=colour, label=row["model"])
    for rect, v in zip(bars, vals):
        ax.annotate(f"{v:.2f}", (rect.get_x() + w / 2, v), (0, 2), textcoords="offset points", ha="center", fontsize=7)
ax.set(
    xticks=xs, xticklabels=["accuracy", "balanced\naccuracy", "recall", "F1"], ylim=(0, 1.2), ylabel="score (5-fold CV)"
)
ax.set_yticks([0, 0.5, 1])
ax.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=3, borderaxespad=0.1, columnspacing=1.0, handletextpad=0.3
)
ax.grid(axis="x", visible=False)
plotlib.save(fg, "d4_paradox")

### Sweep: how rare is too rare? (no slide)

Malignant thinned to a share of the data; 20 random thinnings and splits per row. The baseline accuracy is $1 - $ share and its recall is always 0.

In [ ]:
rows = []
for share in (0.37, 0.2, 0.1, 0.05, 0.02):
    acc_b, acc_t, rec_t, f1_t, rec_l = [], [], [], [], []
    for s in range(20):
        rng = np.random.default_rng(1000 + s)
        Xs, ys = (X, y) if share == 0.37 else thin(X, y, share, rng)
        Xa, Xb, ya, yb = train_test_split(Xs, ys, test_size=0.25, stratify=ys, random_state=s)
        acc_b.append(np.mean(yb == 0))
        mt = metrics(*confusion(yb, tree_model().fit(Xa, ya).predict(Xb)))
        acc_t.append(mt["accuracy"])
        rec_t.append(mt["recall"])
        f1_t.append(mt["f1"])
        rec_l.append(metrics(*confusion(yb, logreg_model().fit(Xa, ya).predict(Xb)))["recall"])
    rows.append({"malignant_share": share, "baseline_accuracy": np.mean(acc_b), "tree_accuracy": np.mean(acc_t),
                 "tree_recall": np.mean(rec_t), "tree_f1": np.mean(f1_t), "logreg_recall": np.mean(rec_l)})  # fmt: skip
sweep_imb = pl.DataFrame(rows)
show(sweep_imb, 3)

## D4e — Comparing two models properly (read in the lab, Part F)

Is the decision tree better than logistic regression, or is the difference just noise?
**One convention for the deck, the lab and the guide.** Model **A = decision tree**, model **B = logistic regression**;
$k = 10$ is the number of folds; $J$ is the **number of scores** (J = 10 for one 10-fold CV, **J = 50 for 5 x 10-fold CV**);
$\delta_j = s_j^{A} - s_j^{B}$ is the score difference on fold or split $j$ (negative when B is better).

* Use **the same folds** for both models and compare the **paired** differences $\delta_j$.
* The scores share training data, so they are not independent and the naive paired t-test is over-confident.
  The **corrected resampled t-test** (Nadeau and Bengio, 2003) replaces $1/J$ by $1/J + n_{test}/n_{train}$ in the variance of the mean:
  $T = \bar{\delta} / \sqrt{(1/J + n_{test}/n_{train})\, s_\delta^2}$, compared with a $t$ distribution with $J - 1$ degrees of freedom.

### Pair the scores: the 10 folds (slide: Pair the Scores; plot `d4_paired`)

In [ ]:
# slide: Pair the Scores (the 10 paired fold scores, J = 10)
slide("Pair the Scores: 10 folds, A = decision tree, B = logistic regression")


def paired_tests(a, b, n_test_over_n_train):
    """Naive paired t-test and corrected resampled t-test (Nadeau and Bengio) on the J paired scores a - b."""
    d = np.asarray(a) - np.asarray(b)
    J = len(d)
    naive_t = d.mean() / np.sqrt(d.var(ddof=1) / J)
    corr_t = d.mean() / np.sqrt(d.var(ddof=1) * (1 / J + n_test_over_n_train))
    return {"J": J, "mean_diff": d.mean(), "s_delta": d.std(ddof=1), "naive_t": naive_t, "naive_p": 2 * stats.t.sf(abs(naive_t), J - 1),
            "corrected_t": corr_t, "corrected_p": 2 * stats.t.sf(abs(corr_t), J - 1)}  # fmt: skip


k = 10
cv10 = StratifiedKFold(n_splits=k, shuffle=True, random_state=SEED)  # the folds of D4c
s_A = cross_val_score(tree_model(), X, y, cv=cv10)
s_B = cross_val_score(logreg_model(), X, y, cv=cv10)
delta = s_A - s_B
show(pl.DataFrame({"fold j": range(1, k + 1), "s_A (tree)": s_A, "s_B (logreg)": s_B, "delta_j = s_A - s_B": delta}), 4)
print(
    f"J = {len(delta)} paired scores; mean difference = {delta.mean():+.4f}; s_delta = {delta.std(ddof=1):.4f}; s of A = {s_A.std(ddof=1):.4f}, of B = {s_B.std(ddof=1):.4f}"
)
r10 = paired_tests(s_A, s_B, 1 / 9)
print(f"naive paired t-test on these J = {r10['J']} scores: T = {r10['naive_t']:.3f}")
print(f"naive p-value = {r10['naive_p']:.4f}")
print(
    f"corrected T on these J = {r10['J']} scores (n_test / n_train = 1/9) = {r10['corrected_t']:.3f}; corrected p-value = {r10['corrected_p']:.4f}  (a different experiment from the J = 50 one below)"
)

### The corrected resampled t-test on 5 x 10-fold CV: J = 50 scores (slide: the corrected resampled t-test; lab Part F)

In [ ]:
# slide: The Corrected Resampled t-Test (J = 50 scores of 5 x 10-fold CV, n_test / n_train = 1/9)
slide("The Corrected Resampled t-Test: J = 50 scores of 5 x 10-fold CV")
rcv = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=SEED)
r_A = cross_val_score(tree_model(), X, y, cv=rcv)
r_B = cross_val_score(logreg_model(), X, y, cv=rcv)
n_test_fold = len(y) / 10
ratio = n_test_fold / (len(y) - n_test_fold)
res = paired_tests(r_A, r_B, ratio)
J = res["J"]
print(f"J = {J} scores (5 repeats x 10 folds); n_test / n_train = {ratio:.4f}")
print(f"mean score: A (tree) {r_A.mean():.4f}, B (logistic regression) {r_B.mean():.4f}")
print(f"mean difference delta-bar = {res['mean_diff']:+.4f}; s_delta = {res['s_delta']:.4f}")
print(f"naive SE = s_delta / sqrt(J) = {res['s_delta'] / np.sqrt(J):.5f}")
print(f"naive T = {res['naive_t']:.2f}")
print(f"naive p-value = {res['naive_p']:.2e}")
print(f"corrected SE = s_delta x sqrt(1/J + n_test/n_train) = {res['s_delta'] * np.sqrt(1 / J + ratio):.5f}")
print(f"corrected T (computed with J = {J}, not with the 10 folds) = {res['corrected_t']:.2f}")
print(f"corrected p-value = {res['corrected_p']:.2e}   ({J - 1} degrees of freedom)")
print(f"the corrected standard error is {np.sqrt(1 / J + ratio) / np.sqrt(1 / J):.2f} times the naive one")

### The failure case: the same learner twice (no slide)

**Null world:** model A and model B are the same learner (a decision tree that looks at 10 random features per split) with two random seeds, so their expected accuracies are equal.
Any "significant" difference is a false positive of the test. Each of 200 experiments draws 150 patients at random, runs 5 x 10-fold CV with the same folds for both models, and tests the $J = 50$ paired scores.
**Real world:** the same experiments with a tree (A) against logistic regression (B), a true difference of a few points; the share of experiments that reject is the **power** of the test.
How often is $p < 0.05$?

The corrected test is **conservative**: in the null world it rejects far less than the 5% it promises, and its power on the real difference is lower than the naive test's. That is the price of not crying wolf: "0% below 0.05" does not mean a perfect test.

In [ ]:
N_EXP, N_SUB, ALPHA = 200, 150, 0.05


def one_experiment(r):
    with threadpool_limits(1):
        return _one_experiment(r)


def _one_experiment(r):
    rng = np.random.default_rng(SEED + r)
    idx = rng.choice(len(y), N_SUB, replace=False)
    Xs, ys = X[idx], y[idx]
    cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=r)
    ratio_r = (N_SUB / 10) / (N_SUB - N_SUB / 10)
    a = cross_val_score(DecisionTreeClassifier(max_features=10, random_state=1), Xs, ys, cv=cv)
    b = cross_val_score(DecisionTreeClassifier(max_features=10, random_state=2), Xs, ys, cv=cv)
    c = cross_val_score(logreg_model(), Xs, ys, cv=cv)
    null = paired_tests(a, b, ratio_r)
    real = paired_tests(a, c, ratio_r)  # A: a tree with random features, B: logistic regression
    return null["naive_p"], null["corrected_p"], real["naive_p"], real["corrected_p"]


p_all = np.array(Parallel(n_jobs=12)(delayed(one_experiment)(r) for r in range(N_EXP)))
p_all = np.nan_to_num(p_all, nan=1.0)  # identical scores give 0 / 0: no evidence of a difference
rates = pl.DataFrame(
    {
        "world": ["null: same learner twice", "real: tree (A) against logistic regression (B)"],
        "naive_rejects": [np.mean(p_all[:, 0] < ALPHA), np.mean(p_all[:, 2] < ALPHA)],
        "corrected_rejects": [np.mean(p_all[:, 1] < ALPHA), np.mean(p_all[:, 3] < ALPHA)],
    }
)
print(f"{N_EXP} experiments, {N_SUB} patients each; share of experiments with p < {ALPHA}:")
show(rates, 3)
print(
    f"null world: a test at the 5% level should reject about {ALPHA:.0%} of the time (naive {rates['naive_rejects'][0]:.1%}, corrected {rates['corrected_rejects'][0]:.1%}: conservative)"
)
print(f"real world, power: naive {rates['naive_rejects'][1]:.1%}, corrected {rates['corrected_rejects'][1]:.1%}")

In [ ]:
# plot d4_paired: left, the paired difference of each fold; right, the p-values of the null world under both tests
fg, axs = plotlib.fig("full", 2.1, ncols=2, gridspec_kw={"width_ratios": [1, 1.15]})
ax = axs[0]
folds_j = np.arange(1, k + 1)
ax.bar(folds_j, delta, color=np.where(delta < 0, ORANGE, BLUE), width=0.7)
ax.axhline(delta.mean(), color="k", ls="--", lw=1)
ax.set(xticks=[1, 5, 10], xlabel="fold $j$", ylabel="$\\delta_j$ = tree $-$ logreg\n($<$ 0: logreg better)")
ax.xaxis.label.set_fontsize(8)
ax.yaxis.label.set_fontsize(8)
ax.set_title(f"mean $\\bar{{\\delta}}$ = {delta.mean():+.3f} (dashed)", fontsize=8)
ax.grid(axis="x", visible=False)
ax = axs[1]
bins = np.linspace(0, 1, 11)
ax.hist(p_all[:, 0], bins=bins, color=RED, alpha=0.8, label=f"naive: {rates['naive_rejects'][0]:.0%} below 0.05")
ax.hist(
    p_all[:, 1],
    bins=bins,
    histtype="step",
    color=BLUE,
    lw=1.6,
    label=f"corrected: {rates['corrected_rejects'][0]:.0%} below 0.05",
)
ax.axvline(ALPHA, color="k", ls=":", lw=1)
ax.set(xlabel="p-value", ylabel="experiments", ylim=(0, ax.get_ylim()[1] * 1.45))
ax.set_title(f"null world: same learner twice,\n{N_EXP} experiments", fontsize=7.5)
ax.legend(loc="upper right", borderaxespad=0.1, handletextpad=0.3, fontsize=7)
fg.subplots_adjust(bottom=0.33, wspace=0.5)
fg.text(0.5, 0.0, f"power on a real difference: naive {rates['naive_rejects'][1]:.1%}, corrected {rates['corrected_rejects'][1]:.1%} (conservative)",
        ha="center", va="bottom", fontsize=7.5)  # fmt: skip
plotlib.save(fg, "d4_paired")

## D4f — McNemar's test: two models, one test set (read in the lab, Part F)

When cross-validation is too expensive and there is one test set, McNemar's test looks only at the patients on which the two models **disagree**.
Model A = decision tree, model B = logistic regression, the test set of D4a (143 patients).

| | B right | B wrong |
|---|---|---|
| **A right** | $n_{11}$ | $n_{10}$ |
| **A wrong** | $n_{01}$ | $n_{00}$ |

$n_{10}$: only A right; $n_{01}$: only B right; $N = n_{10} + n_{01}$ is the number of disagreements. If the models were equally good, the disagreements would split evenly: $n_{10} \approx n_{01}$.
**McNemar (1947):** $\chi^2 = (n_{10} - n_{01})^2 / N$. **Edwards (1948)** subtracts 1 from $|n_{10} - n_{01}|$ (a continuity correction). Both are approximations that need $N \ge 25$;
**for $N < 25$ use the exact binomial test**: $p = 2\,P(\text{Binomial}(N, \tfrac12) \le \min(n_{10}, n_{01}))$, capped at 1.

### The counts and the statistics (slide: McNemar's test; lab Part F; the plot `d4_chi2` is drawn from the out-of-fold counts of the second cell)

In [ ]:
# slide: McNemar's Test (the 2 x 2 counts of the D4a test set; A = decision tree, B = logistic regression)
slide("McNemar's Test on the D4a test set: A = decision tree, B = logistic regression")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
ok_a = tree_model().fit(X_train, y_train).predict(X_test) == y_test  # model A: tree
ok_b = logreg_model().fit(X_train, y_train).predict(X_test) == y_test  # model B: logistic regression
n11, n10 = int(np.sum(ok_a & ok_b)), int(np.sum(ok_a & ~ok_b))
n01, n00 = int(np.sum(~ok_a & ok_b)), int(np.sum(~ok_a & ~ok_b))
print(f"test accuracy: A (tree) = {ok_a.mean():.4f}, B (logistic regression) = {ok_b.mean():.4f}, n = {len(y_test)}")
print(f"{'':24s}{'B right':>9s}{'B wrong':>9s}")
print(f"{'A right':24s}{n11:9d}{n10:9d}")
print(f"{'A wrong':24s}{n01:9d}{n00:9d}")
N = n10 + n01
chi2 = (n10 - n01) ** 2 / N
chi2_cc = (abs(n10 - n01) - 1) ** 2 / N
p_chi2 = stats.chi2.sf(chi2, df=1)
p_chi2_cc = stats.chi2.sf(chi2_cc, df=1)
print(
    f"N = n10 + n01 = {N} disagreements ({'below' if N < 25 else 'at least'} 25: {'use the exact test' if N < 25 else 'the chi-squared approximation is acceptable'})"
)
print(f"chi^2 uncorrected (McNemar 1947) = (n10 - n01)^2 / N = ({n10} - {n01})^2 / {N} = {chi2:.2f}")
print(f"p-value (uncorrected, 1 degree of freedom) = {p_chi2:.4f}")
print(f"chi^2 corrected (Edwards 1948) = (|n10 - n01| - 1)^2 / N = (|{n10} - {n01}| - 1)^2 / {N} = {chi2_cc:.2f}")
print(
    f"p-value (corrected, 1 degree of freedom) = {p_chi2_cc:.4f}; critical value at alpha = 0.05: {stats.chi2.ppf(0.95, 1):.2f}"
)
tail = sum(math.comb(N, j) for j in range(min(n10, n01) + 1))
print(f"exact: P(X <= {min(n10, n01)}) for X ~ Binomial({N}, 0.5) = {tail} / 2^{N} = {tail / 2**N:.5f}")
p_exact = stats.binomtest(n10, N, 0.5).pvalue
print(f"exact two-sided p-value = {min(1.0, 2 * tail / 2**N):.4f}; scipy.stats.binomtest agrees: p = {p_exact:.4f}")

With $N = 9$ disagreements no test can reject: the counts are 4 against 5 and the exact $p$-value is 1. One small test set has little power; the next cell uses the **569 out-of-fold predictions** of the 10-fold CV of D4c (the same folds as the paired scores), where $N \ge 25$ and the three $p$-values can be compared.

In [ ]:
# slide: McNemar's Test (the same 2 x 2 table from the 569 out-of-fold predictions of the 10-fold CV; A = decision tree, B = logistic regression)
slide("McNemar's Test on the out-of-fold predictions of the 10-fold CV: A = decision tree, B = logistic regression")
pred_a = cross_val_predict(tree_model(), X, y, cv=cv10)
pred_b = cross_val_predict(logreg_model(), X, y, cv=cv10)
oa, ob = pred_a == y, pred_b == y
c11, c10 = int(np.sum(oa & ob)), int(np.sum(oa & ~ob))
c01, c00 = int(np.sum(~oa & ob)), int(np.sum(~oa & ~ob))
print(f"out-of-fold accuracy: A (tree) = {oa.mean():.4f}, B (logistic regression) = {ob.mean():.4f}, n = {len(y)}")
print(f"{'':24s}{'B right':>9s}{'B wrong':>9s}")
print(f"{'A right':24s}{c11:9d}{c10:9d}")
print(f"{'A wrong':24s}{c01:9d}{c00:9d}")
NN = c10 + c01
x2 = (c10 - c01) ** 2 / NN
x2c = (abs(c10 - c01) - 1) ** 2 / NN
print(f"N = n10 + n01 = {NN} disagreements ({'below' if NN < 25 else 'at least'} 25)")
print(f"chi^2 uncorrected (McNemar 1947) = ({c10} - {c01})^2 / {NN} = {x2:.2f}; p-value = {stats.chi2.sf(x2, 1):.2e}")
print(
    f"chi^2 corrected (Edwards 1948) = (|{c10} - {c01}| - 1)^2 / {NN} = {x2c:.2f}; p-value = {stats.chi2.sf(x2c, 1):.2e}"
)
print(f"exact two-sided p-value = {stats.binomtest(c10, NN, 0.5).pvalue:.2e}")

In [ ]:
# plot d4_chi2: McNemar on the out-of-fold predictions (N = 42); left the exact null of the disagreements, right the chi-squared approximation
p_ex, p_x2 = stats.binomtest(c10, NN, 0.5).pvalue, stats.chi2.sf(x2, 1)
fg, axs = plotlib.fig("full", 1.7, ncols=2)
ax = axs[0]
js = np.arange(NN + 1)
extreme = (js >= max(c10, c01)) | (js <= min(c10, c01))
ax.bar(js, stats.binom.pmf(js, NN, 0.5), color=np.where(extreme, RED, BLUE), width=0.8)
for obs_j, who in ((c10, "A"), (c01, "B")):
    ax.axvline(obs_j, color=RED, ls="--", lw=1)
    ax.annotate(
        f"{obs_j} ({who})",
        (obs_j, 0.1),
        (-3 if obs_j < NN / 2 else 3, 0),
        textcoords="offset points",
        ha="right" if obs_j < NN / 2 else "left",
        fontsize=7.5,
        color=RED,
    )
ax.set(xlabel=f"disagreements won by A (of {NN})", ylabel="probability if equal", xticks=range(0, NN + 1, 7))
ax.set_title(f"exact: $p$ = {p_ex:.1e}\n(red: as extreme as {c10} against {c01})", fontsize=8)
ax.grid(axis="x", visible=False)
ax = axs[1]
xs = np.linspace(0.05, x2 + 3, 300)
ax.plot(xs, stats.chi2.pdf(xs, 1), color=BLUE)
ax.fill_between(xs[xs >= x2], stats.chi2.pdf(xs[xs >= x2], 1), color=RED, alpha=0.7, lw=0)
ax.axvline(stats.chi2.ppf(0.95, 1), color="k", ls=":", lw=1)
ax.annotate(
    "critical 3.84", (stats.chi2.ppf(0.95, 1), 0.6), (4, 0), textcoords="offset points", ha="left", fontsize=7.5
)
ax.annotate(
    f"$\\chi^2$ = {x2:.2f}", (x2, 0.02), (-4, 14), textcoords="offset points", ha="right", fontsize=8, color=RED
)
ax.set(xlabel="$\\chi^2$ (1 degree of freedom)", ylabel="density", ylim=(0, 0.9))
ax.set_title(f"approximation (large $N$)\n$p$ = {p_x2:.1e}", fontsize=8)
plotlib.save(fg, "d4_chi2")

### Failure case: few disagreements (no slide)

The chi-squared approximation needs about 25 disagreements. With fewer, use the exact binomial test. The last block repeats the comparison on 50 random splits.

In [ ]:
rows = []
for a, b in [(5, 0), (8, 2), (10, 1), (3, 3), (20, 10), (30, 15), (60, 40)]:
    nn = a + b
    c_u, c_c = (a - b) ** 2 / nn, (abs(a - b) - 1) ** 2 / nn
    rows.append({"n10": a, "n01": b, "N": nn, "chi2": c_u, "p_chi2": stats.chi2.sf(c_u, 1), "chi2_cc": c_c, "p_chi2_cc": stats.chi2.sf(c_c, 1),
                 "p_exact": stats.binomtest(a, nn, 0.5).pvalue})  # fmt: skip
show(pl.DataFrame(rows), 4)

rows = []
for s in range(50):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, stratify=y, random_state=s)
    ra = tree_model().fit(Xa, ya).predict(Xb) == yb
    rb = logreg_model().fit(Xa, ya).predict(Xb) == yb
    a, b = int(np.sum(ra & ~rb)), int(np.sum(~ra & rb))
    rows.append({"split": s, "n10": a, "n01": b, "p_exact": stats.binomtest(a, a + b, 0.5).pvalue if a + b else 1.0})
splits = pl.DataFrame(rows)
print(
    f"50 random splits: McNemar (exact) rejects 'equal' at 5% in {splits.filter(pl.col('p_exact') < 0.05).height} splits"
)
print(
    f"discordant pairs per split: median n10 = {splits['n10'].median()} (only the tree right), median n01 = {splits['n01'].median()} (only logistic regression right)"
)

## D4g — Hypothesis testing from scratch

**Null hypothesis** $H_0$: nothing special happens (the coin is fair, the model is no better than guessing). The **p-value** is the probability, if $H_0$ were true,
of a result at least as extreme as the one observed. The **significance level** $\alpha = 0.05$ is the false positive rate of the test that we accept in advance.
A p-value is **not** the probability that $H_0$ is true.

### 10 tosses, 9 heads (slides: Is the Difference Real? Hypothesis Testing; Worked Example: a Coin and a Classifier; plot `d4_null`)

In [ ]:
# slide: Worked Example: a Coin and a Classifier
slide("Worked Example: a Coin and a Classifier")
rng = np.random.default_rng(SEED)
n_toss, heads = 10, 9
counts = [math.comb(n_toss, j) for j in range(n_toss + 1)]
print(
    f"number of outcomes of {n_toss} tosses: 2^{n_toss} = {2**n_toss}; ways to get exactly j heads, j = 0..{n_toss}: {counts}"
)
ways = sum(counts[heads:])
p_one = ways / 2**n_toss
print(
    f"P(at least {heads} heads) = ({counts[9]} + {counts[10]}) / {2**n_toss} = {ways} / {2**n_toss} = {p_one:.5f}  (one-sided p-value)"
)
print(f"two-sided p-value (9 or more heads, or 1 or fewer) = {2 * p_one:.5f}")
print(
    f"scipy.stats.binomtest: one-sided p = {stats.binomtest(heads, n_toss, 0.5, alternative='greater').pvalue:.5f}, two-sided p = {stats.binomtest(heads, n_toss, 0.5).pvalue:.5f}"
)
sim = rng.binomial(n_toss, 0.5, size=1_000_000)
print(f"simulation: 1 000 000 experiments with a fair coin, share with >= 9 heads = {np.mean(sim >= heads):.5f}")
print(
    f"false positive rate of the rule 'reject H0 if heads >= 9' = {p_one:.4f}; with 'heads >= 8' it would be {sum(counts[8:]) / 2**n_toss:.4f} (above alpha = 0.05)"
)
print("decision at alpha = 0.05: p = %.4f < 0.05, reject H0 (the coin looks biased)" % p_one)

In [ ]:
# plot d4_null
fg, ax = plotlib.fig("full", 1.6)
js = np.arange(n_toss + 1)
pmf = np.array(counts) / 2**n_toss
ax.bar(js, pmf, color=np.where(js >= heads, RED, BLUE), width=0.7)
for j in js:
    ax.annotate(f"{counts[j]}", (j, pmf[j]), (0, 2), textcoords="offset points", ha="center", fontsize=7.5)
ax.annotate(f"$p$ = {p_one:.3f}", (9.5, 0.06), (0, 12), textcoords="offset points", ha="center", color=RED, fontsize=8)
ax.set(
    xlabel="heads in 10 tosses of a fair coin (bar label: ways out of 1024)",
    ylabel="probability under $H_0$",
    xticks=js,
    ylim=(0, 0.3),
)
ax.grid(axis="x", visible=False)
plotlib.save(fg, "d4_null")

### The same test for a classifier (slide: Worked Example: a Coin and a Classifier)

$H_0$: the tree is no better than always answering "benign". Under $H_0$ each patient is classified correctly with probability equal to the share of benign patients.

In [ ]:
# slide: Worked Example: a Coin and a Classifier (the classifier)
slide("a Classifier against the baseline")
share_benign = float(np.mean(y_train == 0))
n_ok = int(np.sum(ok_a))
bt = stats.binomtest(n_ok, len(y_test), share_benign, alternative="greater")
print(
    f"tree: {n_ok} of {len(y_test)} test patients correct; the baseline says benign to everyone: accuracy {np.mean(y_test == 0):.4f} on this test set"
)
print(f"H0: P(correct) = {share_benign:.4f} (share of benign in the training set, the accuracy of the baseline)")
print(f"p-value of the tree against the baseline (one-sided, {n_ok} of {len(y_test)} correct) = {bt.pvalue:.3e}")
print(
    f"9 correct out of 10 against P(correct) = 0.5: p = {stats.binomtest(9, 10, 0.5, alternative='greater').pvalue:.4f}; 90 correct out of 100 against 0.5: p = {stats.binomtest(90, 100, 0.5, alternative='greater').pvalue:.2e}"
)

### Permutation tests (no slide; lab challenge)

A permutation test builds the null distribution by shuffling, with no formula.
**(a)** Is the tree better than chance? Shuffle the training labels (the features then carry no information), retrain, test on the true test labels.
**(b)** Is one model better than the other on this test set (A = tree, B = logistic regression)? Per patient the difference of correctness is $-1$, $0$ or $+1$; under $H_0$ its sign is arbitrary, so flip signs at random.

In [ ]:
rng = np.random.default_rng(SEED)
obs = ok_a.mean()
null_acc = np.array([tree_model().fit(X_train, rng.permutation(y_train)).score(X_test, y_test) for _ in range(200)])
p_a = (1 + np.sum(null_acc >= obs)) / (1 + len(null_acc))
print(
    f"(a) tree test accuracy = {obs:.4f}; accuracy after shuffling the training labels: mean = {null_acc.mean():.4f}, max = {null_acc.max():.4f} (200 shuffles)"
)
print(f"    permutation p-value = (1 + {int(np.sum(null_acc >= obs))}) / (1 + 200) = {p_a:.4f}")
d = ok_a.astype(int) - ok_b.astype(int)
flips = rng.choice([-1, 1], size=(100_000, len(d)))
null_d = (flips * d).mean(axis=1)
p_b = (1 + np.sum(np.abs(null_d) >= abs(d.mean()))) / (1 + len(null_d))
print(
    f"(b) mean of the per-patient differences = {d.mean():+.4f} (n10 - n01 = {n10 - n01} over {len(d)} patients); sign-flip permutation p-value (two-sided, 100 000 flips) = {p_b:.4f}"
)
print(f"    compare McNemar exact p = {p_exact:.4f}: the two tests ask the same question")

In [ ]:
# plot d4_perm (extra): the null distribution of (a)
fg, ax = plotlib.fig("col", 1.6)
ax.hist(null_acc, bins=np.arange(0.5, 1.0, 0.02), color=GREY, edgecolor="white", lw=0.5)
ax.axvline(obs, color=RED, lw=1.4)
ax.annotate(
    f"tree: {obs:.3f}",
    (obs, ax.get_ylim()[1]),
    (-4, -8),
    textcoords="offset points",
    ha="right",
    color=RED,
    fontsize=7.5,
)
ax.set(xlabel="test accuracy, training labels shuffled", ylabel="shuffles", xlim=(0.5, 1.0))
plotlib.save(fg, "d4_perm")

**Takeaways**

1. Never evaluate on training data; one split is one draw (D4a, D4b).
2. A score is an estimate with uncertainty: report the mean and its interval (D4c); the interval depends on $n$, on $k$ and on the problem.
3. Beat a baseline, with a metric that matches the goal (D4d).
4. Compare models on the same folds with a test that accounts for the overlap (corrected t-test), or on the same test set with McNemar, exact when $N < 25$ (D4e, D4f); the naive test cries wolf, the corrected one is conservative.
5. A p-value is the probability of data at least this extreme if nothing is going on, not the probability that nothing is going on (D4g).